# 01 — PINN for ODE

**Module 3 · Week 5 · TA-3**

This notebook covers:
- What is a physics residual
- Computing derivatives with `torch.autograd.grad`
- PINN loss = physics residual + initial condition
- Comparing PINN prediction vs analytic solution

**Target ODE:** Exponential decay
```
du/dt = -k * u,   u(0) = 1,   k = 1.0
Analytic solution: u(t) = exp(-t)
```

---

In [ ]:
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
import numpy as np

torch.manual_seed(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
k = 1.0  # decay constant
T_end = 5.0  # solve on t in [0, T_end]
print(f'Device: {device}')

## 1. Define the PINN Model

In [ ]:
class PINN(nn.Module):
    def __init__(self, hidden: int = 64, n_layers: int = 3):
        super().__init__()
        # --- TODO: Define MLP ---
        # Input: 1 (time t)
        # Hidden: n_layers layers with tanh activation
        # Output: 1 (u(t))
        # Note: tanh recommended for smooth ODE solutions
        pass

    def forward(self, t):
        pass

model = PINN().to(device)

## 2. Collocation Points and IC Points

In [ ]:
N_colloc = 1000  # physics collocation points

# Collocation points: random in [0, T_end]
t_colloc = torch.rand(N_colloc, 1) * T_end
t_colloc = t_colloc.to(device).requires_grad_(True)

# Initial condition point: t = 0, u = 1
t_ic = torch.zeros(1, 1, device=device)
u_ic = torch.ones(1, 1, device=device)

print(f'Collocation points: {t_colloc.shape}')
print(f'IC point: t={t_ic.item()}, u={u_ic.item()}')

## 3. Physics Residual

In [ ]:
def physics_residual(model, t, k):
    """Compute ODE residual: du/dt + k*u = 0."""
    u = model(t)
    # --- TODO: Compute du/dt using torch.autograd.grad ---
    # du_dt = torch.autograd.grad(u, t, grad_outputs=..., create_graph=True)[0]
    # residual = du_dt + k * u
    # return residual
    pass

## 4. Training Loop

In [ ]:
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
N_STEPS = 10000
W_PHYSICS = 1.0
W_IC = 10.0  # higher weight on IC to enforce initial condition

loss_history = {'total': [], 'physics': [], 'ic': []}

for step in range(N_STEPS):
    optimizer.zero_grad()

    # --- TODO: Compute total loss ---
    # 1. Physics loss: mean(residual**2) at collocation points
    # 2. IC loss: MSE(model(t_ic), u_ic)
    # 3. total = W_PHYSICS * physics_loss + W_IC * ic_loss

    if (step + 1) % 1000 == 0:
        print(f'Step {step+1}/{N_STEPS}  '
              f'Total: {loss_history["total"][-1]:.6f}  '
              f'Physics: {loss_history["physics"][-1]:.6f}  '
              f'IC: {loss_history["ic"][-1]:.6f}')

## 5. Compare PINN vs Analytic Solution

In [ ]:
t_test = torch.linspace(0, T_end, 500).reshape(-1, 1).to(device)
with torch.no_grad():
    u_pred = model(t_test).cpu().numpy()
t_np = t_test.cpu().numpy()
u_analytic = np.exp(-k * t_np)

plt.figure(figsize=(8, 4))
plt.plot(t_np, u_analytic, 'b-', label='Analytic: exp(-t)', linewidth=2)
plt.plot(t_np, u_pred, 'r--', label='PINN prediction', linewidth=2)
plt.xlabel('t')
plt.ylabel('u(t)')
plt.title('PINN vs Analytic Solution (du/dt = -ku)')
plt.legend()
plt.grid(True)
plt.show()

mse = np.mean((u_pred - u_analytic)**2)
print(f'MSE vs analytic: {mse:.6f}')

## 6. Plot Loss History

In [ ]:
# --- TODO: Plot physics loss and IC loss over training steps ---


## 7. Ablation: Loss Weights

*(Complete as part of Week 5 assignment)*

Rerun training with different weight settings and fill in the table:

| W_PHYSICS | W_IC | Final MSE vs analytic |
|-----------|------|----------------------|
| 1.0 | 1.0 | |
| 1.0 | 10.0 | |
